In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.12 The Quantum Harmonic Oscillator

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.12",
    title="The Quantum Harmonic Oscillator",
    blurb="The most important system in physics. Every potential near a minimum is "
    "a parabola, so the oscillator is the universal first approximation to almost "
    "everything — and it can be solved three ways that must agree: by an algebra of "
    "raising and lowering operators that produces the whole spectrum without a single "
    "derivative, by Hermite functions, and on the grid. We meet its zero-point "
    "energy, contrast it sharply with the classical oscillator, and build the "
    "coherent states that oscillate like a classical particle while staying as sharp "
    "as the uncertainty principle allows.",
    difficulty="advanced",
    estimate="170–210 min",
)

## Notebook overview

If one had to pick a single system to understand deeply, it would be this one. The harmonic oscillator
is the most important problem in physics, and for a reason that is almost embarrassingly simple: *every*
smooth potential, near a stable minimum, is a parabola. Taylor-expand any well around its bottom and
the leading term is $\tfrac12V''(x_0)(x-x_0)^2$ — a harmonic oscillator. So the oscillator's solution
is the universal first approximation to vibrating molecules, the phonons of a crystal, and the modes of
the electromagnetic field (each mode a quantum oscillator, its quantum the photon). Master it and you
hold the leading description of an enormous swath of nature.

It earns a deep treatment because it can be solved **three independent ways that must agree**, and
seeing them coincide is the structural lesson of the notebook. First, **algebraically**: from the
single commutator $[a,a^{\dagger}]=1$, a pair of **ladder operators** generates the entire spectrum —
equally-spaced levels $E_n=\hbar\omega(n+\tfrac12)$ — with *no differential equation at all*. This is
the Sakurai gem, and the same algebra returns for angular momentum ([§6.14](angular-momentum-algebra.ipynb)) and, in Volume VII, to count
photons. Second, **analytically**: solving the differential equation gives the **Hermite functions**,
whose ground state is exactly the minimum-uncertainty Gaussian of [§6.9](position-representation.ipynb). Third, **numerically**: the
[§6.10](schrodinger-on-a-computer.ipynb) grid solver, reused. All three give the same spectrum and the same states.

Along the way we meet the features that make the *quantum* oscillator differ from Volume V's classical
one — a discrete, equally-spaced ladder and a nonzero **zero-point energy** $\hbar\omega/2$ (it cannot
sit at rest, forced by the uncertainty principle), against continuous energy and equipartition — and
we watch the **correspondence principle** reconcile them at high quantum number. We finish with
**coherent states**, the eigenstates of the lowering operator: the [§6.9](position-representation.ipynb) Gaussian set in motion, which
oscillate *rigidly* like a classical particle while staying minimum-uncertainty for all time — the
cleanest bridge from quantum to classical motion, and the state of a laser field.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the ladder matrices via `numpy.diag` with $\sqrt n$ entries,
`numpy.linalg.eigh` for the grid, `numpy.polynomial.hermite.hermval` for the Hermite polynomials, and
`math.factorial` for the normalizations.

> **Two numerical notes.** (i) `numpy.math` was **removed in NumPy 2.0** — we use `math.factorial`
> (`scipy.special.factorial` is the array alternative), never `np.math.factorial`. (ii) The ladder
> matrices live in a **truncated** number basis of $N_{\max}$ levels, so $[a,a^{\dagger}]=1$ holds only
> *away from* the top row and column (the truncation breaks it there); we verify it on the interior
> block and say so.
>
> **Conventions and scope.** $\hbar=m=\omega=1$, so $E_n=n+\tfrac12$. The number basis is truncated at
> $N_{\max}=40$; the grid solution uses the [§6.10](schrodinger-on-a-computer.ipynb) finite-difference solver (eigenfunctions normalized by
> $\sqrt{dx}$). Squeezed states and the oscillator-as-photon are horizons (Volume VII). See Sakurai &
> Napolitano (§2.3, the operator method); Griffiths (the Hermite solution); and Notebooks [§6.9](position-representation.ipynb) (the
> Gaussian, $[x,p]=i\hbar$), [§6.10](schrodinger-on-a-computer.ipynb) (the grid solver), [§6.6](pauli-uncertainty.ipynb) (uncertainty), and Volume V (the classical
> oscillator, equipartition).

## Theory in brief

### Why the oscillator is everywhere

Any smooth potential near a stable minimum is, by Taylor expansion, a parabola,

```{math}
:label: eq-why-oscillator
V(x)\approx\tfrac12 V''(x_0)\,(x-x_0)^2=\tfrac12 m\omega^2(x-x_0)^2,\qquad H=\frac{p^2}{2m}+\tfrac12 m\omega^2x^2 ,
```

so the oscillator is the universal leading description of molecules, phonons, and field modes.

### The ladder operators — the algebraic solution

The algebraic route begins with a factorization: up to an additive constant, $H$ is a product of
two first-order factors in $x$ and $p$. Define the **lowering** and **raising** operators

```{math}
:label: eq-ladder
a=\sqrt{\tfrac{m\omega}{2\hbar}}\Big(x+\tfrac{ip}{m\omega}\Big),\quad [a,a^{\dagger}]=1,\quad H=\hbar\omega\big(a^{\dagger}a+\tfrac12\big)=\hbar\omega\big(N+\tfrac12\big),\quad N=a^{\dagger}a ,
```

with $N$ the **number operator**. The entire spectrum follows from the algebra alone: $a$ lowers the
energy by $\hbar\omega$ and $a^{\dagger}$ raises it, the ground state obeys $a|0\rangle=0$, and the
ladder is $a|n\rangle=\sqrt n\,|n-1\rangle$, $a^{\dagger}|n\rangle=\sqrt{n+1}\,|n+1\rangle$, $N|n\rangle
=n|n\rangle$, giving $E_n=\hbar\omega(n+\tfrac12)$ — **equally spaced**. (The same method returns for
angular momentum, [§6.14](angular-momentum-algebra.ipynb), and quantum fields.)

### The zero-point energy

The ladder bottoms out but the energy does not: the ground state is annihilated by $a$, so
$N|0\rangle=0$, and the $\tfrac12$ in $H=\hbar\omega(N+\tfrac12)$ survives,

```{math}
:label: eq-zero-point
E_0=\tfrac12\hbar\omega\ne0 ,
```

the oscillator *cannot* sit at rest. This is the uncertainty principle ([§6.6](pauli-uncertainty.ipynb), [§6.9](position-representation.ipynb)): localizing the
particle costs momentum spread, and minimizing $\langle p^2\rangle/2m+\tfrac12 m\omega^2\langle x^2
\rangle$ subject to $\Delta x\,\Delta p\ge\hbar/2$ gives exactly $\hbar\omega/2$. (Zero-point energy is
real — the Casimir effect, liquid helium; flagged, not developed.)

### The analytic solution: Hermite functions

Solving the differential equation directly, by the power-series method carried out in full in
Griffiths, gives the **Hermite functions**,

```{math}
:label: eq-hermite
\psi_n(x)=\Big(\tfrac{m\omega}{\pi\hbar}\Big)^{1/4}\frac{1}{\sqrt{2^n n!}}\,H_n(\xi)\,e^{-\xi^2/2},\qquad \xi=\sqrt{\tfrac{m\omega}{\hbar}}\,x ,
```

with $H_n$ the Hermite polynomials (`numpy.polynomial.hermite.hermval`, `math.factorial`). The ground
state is the Gaussian of [§6.9](position-representation.ipynb) — the minimum-uncertainty state.

### Three routes agree

The three solutions arrive by entirely different means (commutators, a differential equation, a
matrix), but they address one and the same eigenvalue problem, so their agreement is a theorem
rather than a coincidence:

```{math}
:label: eq-three-routes
\{E_n,\psi_n\}_{\text{ladder algebra}}=\{E_n,\psi_n\}_{\text{Hermite}}=\{E_n,\psi_n\}_{\text{grid}} ,
```

the structural lesson: abstract algebra, special functions, and brute-force diagonalization are three
views of one problem.

### Classical versus quantum, and correspondence

The **quantum** oscillator has discrete, equally-spaced levels and a nonzero ground-state energy;
Volume V's **classical** oscillator has continuous energy, obeys equipartition $\langle E\rangle=kT$,
and can rest at $E=0$. They reconcile at high $n$,

```{math}
:label: eq-classical-quantum
|\psi_n(x)|^2\ \xrightarrow{\ n\to\infty,\ \text{smoothed}\ }\ P_{\text{cl}}(x)=\frac{1}{\pi\sqrt{A^2-x^2}} ,
```

the classical distribution (a classical oscillator lingers near its turning points $\pm A$, where it
moves slowly): quantum mechanics contains classical mechanics in the large-$n$ limit.

### Coherent states — the most classical quantum state

The eigenstates of the **lowering** operator,

```{math}
:label: eq-coherent
a|\alpha\rangle=\alpha|\alpha\rangle,\qquad |\alpha\rangle=e^{-|\alpha|^2/2}\sum_n\frac{\alpha^n}{\sqrt{n!}}|n\rangle,\qquad \langle x\rangle(t)=\sqrt{\tfrac{2\hbar}{m\omega}}\,\mathrm{Re}\big(\alpha e^{-i\omega t}\big) ,
```

are **coherent states** — displaced ground states with $\langle N\rangle=|\alpha|^2$. They are
minimum-uncertainty ($\Delta x\,\Delta p=\hbar/2$) and *stay* so for all time, and they oscillate
rigidly like a classical particle: the [§6.9](position-representation.ipynb) Gaussian set in motion without spreading — the bridge from
quantum to classical dynamics, and the state of a laser. Sakurai & Napolitano close §2.3 with their
construction and properties.

## Setup

In [ ]:
from math import factorial  # NOT numpy.math (removed in NumPy 2.0)

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from numpy.polynomial.hermite import hermval

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

HBAR = 1.0
MASS = 1.0
OMEGA = 1.0
N_MAX = 40  # truncation of the number basis


def ladder_operators(n_max):
    """The oscillator's ladder operators $a,a^{\\dagger}$, number operator $N$, and Hamiltonian $H$ {eq}`eq-ladder`.

    In the truncated number basis $\\{|0\\rangle,\\dots,|n_{\\max}-1\\rangle\\}$, the lowering operator $a$
    has $\\sqrt n$ on the upper off-diagonal (`numpy.diag(numpy.sqrt(numpy.arange(1, n_max)), 1)`, since
    $a|n\\rangle=\\sqrt n|n-1\\rangle$), and $a^{\\dagger}$ is its transpose. Then $N=a^{\\dagger}a$ and
    $H=\\hbar\\omega(N+\\tfrac12)$. **Truncation caveat:** $[a,a^{\\dagger}]=1$ holds only away from the top
    row/column, where the missing $|n_{\\max}\\rangle$ breaks it.

    Returns
    -------
    a, a_dag, N, H : numpy.ndarray
        The lowering, raising, number, and Hamiltonian matrices.
    """
    a = np.diag(np.sqrt(np.arange(1, n_max)), 1)
    a_dag = a.conj().T
    number = a_dag @ a
    hamiltonian = HBAR * OMEGA * (number + 0.5 * np.eye(n_max))
    return a, a_dag, number, hamiltonian


def hermite_eigenfunction(n, x):
    """The $n$-th oscillator eigenfunction, a Hermite function {eq}`eq-hermite`.

    $\\psi_n(x)=(m\\omega/\\pi\\hbar)^{1/4}(2^n n!)^{-1/2}H_n(\\xi)e^{-\\xi^2/2}$ with $\\xi=\\sqrt{m\\omega/
    \\hbar}\\,x$, evaluated with `numpy.polynomial.hermite.hermval` and `math.factorial` (cast to ``float``
    so `numpy.sqrt` does not choke on a large Python integer).
    """
    xi = np.sqrt(MASS * OMEGA / HBAR) * x
    coef = np.zeros(n + 1)
    coef[n] = 1.0
    norm = (MASS * OMEGA / (np.pi * HBAR)) ** 0.25 / np.sqrt(2**n * float(factorial(n)))
    return norm * hermval(xi, coef) * np.exp(-(xi**2) / 2)


def coherent_state(alpha, n_max):
    """The coherent state $|\\alpha\\rangle$ in the number basis {eq}`eq-coherent`.

    $|\\alpha\\rangle=e^{-|\\alpha|^2/2}\\sum_n(\\alpha^n/\\sqrt{n!})|n\\rangle$, with `math.factorial`
    (cast to ``float`` to avoid overflow in `numpy.sqrt`). Re-normalized for the basis truncation. It is
    the eigenstate of the lowering operator, $a|\\alpha\\rangle=\\alpha|\\alpha\\rangle$.
    """
    coeffs = np.array(
        [alpha**n / np.sqrt(float(factorial(n))) for n in range(n_max)], dtype=complex
    )
    coeffs *= np.exp(-abs(alpha) ** 2 / 2)
    return coeffs / np.linalg.norm(coeffs)


def solve(x, V):
    """The §6.10 finite-difference grid solver, reused: energies and normalized stationary states {eq}`eq-three-routes`.

    The $(1,-2,1)/dx^2$ kinetic stencil plus $\\mathrm{diag}\\,V$, diagonalized by ``numpy.linalg.eigh``;
    columns divided by $\\sqrt{dx}$. Built and validated in §6.10 — used here as a tool.
    """
    n = len(x)
    dx = x[1] - x[0]
    lap = (
        np.diag(-2 * np.ones(n))
        + np.diag(np.ones(n - 1), 1)
        + np.diag(np.ones(n - 1), -1)
    ) / dx**2
    energies, states = np.linalg.eigh(-(HBAR**2 / (2 * MASS)) * lap + np.diag(V))
    return energies, states / np.sqrt(dx)


def expval(op, psi):
    """The expectation $\\langle\\psi|\\hat O|\\psi\\rangle$ in the number basis (``numpy.vdot``), real part."""
    return np.real(np.vdot(psi, op @ psi))


# the number-basis position and momentum operators, x = √(ℏ/2mω)(a+a†), p = i√(ℏmω/2)(a†−a)
A_OP, A_DAG, N_OP, H_OP = ladder_operators(N_MAX)
X_OP = np.sqrt(HBAR / (2 * MASS * OMEGA)) * (A_OP + A_DAG)
P_OP = 1j * np.sqrt(HBAR * MASS * OMEGA / 2) * (A_DAG - A_OP)

## Exercise 1 — The ladder operators and the algebra

Construct the ladder operators $a,a^{\dagger}$ in the number basis and verify the defining
algebra: $[a,a^{\dagger}]=1$ (away from the truncation), and the number operator $N=a^{
\dagger}a=\mathrm{diag}(0,1,2,\dots)$ {eq}`eq-ladder`.

1. Build $a$ with $\sqrt n$ on the upper off-diagonal (`numpy.diag(numpy.sqrt(numpy. arange(1,
   N_max)), 1)`) and $a^{\dagger}=a^{\mathsf T}$ (the `ladder_operators` helper).
2. Form the commutator $[a,a^{\dagger}]=aa^{\dagger}-a^{\dagger}a$ and confirm it equals the
   identity on the **interior** block (the top row/column is spoiled by the missing
   $|N_{\max}\rangle$ — the truncation caveat).
3. Form $N=a^{\dagger}a$ and confirm it is $\mathrm{diag}(0,1,2,\dots)$.
4. Build $H=\hbar \omega(N+\tfrac12)$. Frame: the algebra, not a differential equation.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    comm_ok and number_ok,
    "the ladder algebra [a,a†]=1 (away from the truncation) generates the number operator N=a†a=diag(0,1,2,…)",
)

## Exercise 2 — The spectrum from the algebra

Derive the oscillator spectrum from the ladder operators alone, with no differential equation:
confirm $H=\hbar\omega(N+\tfrac12)$ has eigenvalues $E_n=\hbar\omega(n+\tfrac12)$, equally spaced
by $\hbar\omega$ {eq}`eq-ladder`.

1. Recall the logic: $a|0\rangle=0$ defines the ground state; $a^{\dagger}$ raises the energy by
   $\hbar\omega$ and $a$ lowers it; so the spectrum is the ladder $E_n=\hbar\omega(n+\tfrac12)$.
2. Read the energies off the diagonal of $H$ (it is already diagonal in the number basis).
3. Confirm they equal $\hbar\omega(n+\tfrac12)$ and that the spacing is exactly $\hbar\omega$.
4. Verify $a|0\rangle=0$ on the basis vector $|0\rangle$. Frame: the whole spectrum without
   solving a differential equation — the power that returns for angular momentum ([§6.14](angular-momentum-algebra.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    spectrum,
    expected,
    "the spectrum Eₙ = ℏω(n+½), equally spaced by ℏω, follows from the ladder algebra alone",
    rtol=1e-12,
)

## Exercise 3 — The grid solution agrees

Solve the oscillator on the [§6.10](schrodinger-on-a-computer.ipynb) grid ($V=\tfrac12 m\omega^2x^2$) and confirm its spectrum matches
the algebraic one, $E_n=\hbar\omega(n+\tfrac12)$ with equal spacing {eq}`eq-three-routes`.

1. Build $V=\tfrac12 m\omega^2x^2$ on a box and solve with the [§6.10](schrodinger-on-a-computer.ipynb) `solve` helper
   (`numpy.linalg.eigh`).
2. Compare the lowest six levels to $\hbar\omega(n+\tfrac12)$.
3. Confirm the spacing is $\hbar\omega$.
4. Note that the highest levels converge later (the [§6.10](schrodinger-on-a-computer.ipynb) resolution caveat — their wave functions
   are wider and wigglier). Frame: the algebra and brute-force diagonalization agree.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    E_grid[:6],
    expected[:6],
    "the grid spectrum matches ℏω(n+½) with equal spacing ℏω — the algebra and diagonalization agree",
    atol=1e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The eigenfunctions are Hermite functions

Confirm the grid eigenfunctions are the analytic **Hermite functions** $\psi_n(x)=
(m\omega/\pi\hbar)^{1/4}(2^n n!)^{-1/2}H_n(\xi)e^{-\xi^2/2}$, matching to high precision, and note
the ground state is the Gaussian of [§6.9](position-representation.ipynb) {eq}`eq-hermite`.

1. Build $\psi_n(x)$ with the `hermite_eigenfunction` helper (`numpy.polynomial.hermite. hermval`
   and `math.factorial` — *not* `np.math`).
2. Compare to the grid eigenfunctions from Exercise 3, fixing the sign (eigenvectors are defined
   up to a sign — flip so the overlap is positive).
3. Confirm agreement to within the grid's discretization error.
4. Note $\psi_0(x)\propto e^{-m\omega x^2/2\hbar}$ is the minimum-uncertainty Gaussian of [§6.9](position-representation.ipynb).
   Frame: the analytic solution, matching the grid — the second of the three routes.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    max(hermite_errors),
    0.0,
    "the eigenfunctions are the Hermite functions Hₙ(ξ)e^{−ξ²/2} (matching the grid)",
    atol=1e-3,
)

## Exercise 5 — The zero-point energy

Explain and verify why the ground state has $E_0=\hbar\omega/2\ne0$: compute $\langle x^2\rangle$
and $\langle p^2\rangle$ in $|0\rangle$, show $\langle H\rangle=\hbar\omega/2$, and confirm the
ground state is minimum-uncertainty — the zero-point energy is the uncertainty principle made
energetic {eq}`eq-zero-point`.

1. In the number basis, build $x$ and $p$ as $x=\sqrt{\hbar/2m\omega}(a+a^{\dagger})$ and
   $p=i\sqrt{\hbar m\omega/2}(a^{\dagger}-a)$ (the `X_OP`, `P_OP` of the setup).
2. Compute $\langle x^2 \rangle$ and $\langle p^2\rangle$ in $|0\rangle$ (`expval`).
3. Form $\langle H\rangle=\langle p^2 \rangle/2m+\tfrac12 m\omega^2\langle x^2\rangle$ and confirm
   it is $\hbar\omega/2$.
4. Confirm $\Delta x\,\Delta p=\hbar/2$ (minimum uncertainty, since $\langle x\rangle=\langle
   p\rangle=0$). Frame: the oscillator cannot be at rest — localizing it costs kinetic energy
   ([§6.6](pauli-uncertainty.ipynb), [§6.9](position-representation.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [E0, dx_ground * dp_ground],
    [HBAR * OMEGA / 2, HBAR / 2],
    "the zero-point energy E₀=ℏω/2 is forced by the uncertainty principle (the ground state is minimum-uncertainty)",
    rtol=1e-6,
)

## Exercise 6 — Classical versus quantum, and the correspondence principle

Contrast the quantum oscillator with Volume V's classical one, and show they reconcile at high
$n$: the smoothed high-$n$ probability density $|\psi_n(x)|^2$ approaches the classical
distribution $P_{\text{cl}}(x)\propto1/\sqrt{A^2-x^2}$, which lingers near the turning points
{eq}`eq-classical-quantum`.

1. Tabulate the differences: the quantum oscillator has *discrete, equally-spaced* levels and a
   *nonzero* ground-state energy; the classical one (Volume V) has *continuous* energy, obeys
   equipartition $\langle E\rangle=kT$, and can rest at $E=0$.
2. Compute the high-$n$ density $|\psi_n(x)|^2$ for $n\approx30$ with the `hermite_eigenfunction`
   helper.
3. Overlay the classical distribution $P_{\text{cl}}(x)=1/(\pi\sqrt{A^2-x^2})$ with turning point
   $A=\sqrt{2E_n/m\omega^2}$ (a classical oscillator moves slowest, and so is most likely found,
   near $\pm A$).
4. Confirm the *smoothed* quantum density tracks the classical envelope. Frame: quantum mechanics
   contains classical mechanics as $n\to\infty$ (the correspondence principle).

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    rel_diff < 0.2,
    "the correspondence principle: the smoothed high-n |ψₙ|² approaches the classical distribution 1/√(A²−x²) — quantum becomes classical at large n",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Coherent states

Construct a coherent state and verify it is a minimum-uncertainty eigenstate of the **lowering**
operator: $a|\alpha\rangle=\alpha|\alpha\rangle$, $\langle N\rangle=|\alpha|^2$, and $\Delta
x\,\Delta p=\hbar/2$ {eq}`eq-coherent`.

1. Build $|\alpha\rangle=e^{-|\alpha|^2/2}\sum_n(\alpha^n/\sqrt{n!})|n\rangle$ with the
   `coherent_state` helper (`math.factorial`, cast to `float`).
2. Verify it is an eigenstate of the lowering operator, $a|\alpha\rangle=\alpha|\alpha\rangle$
   (`numpy.allclose`).
3. Confirm $\langle N \rangle=|\alpha|^2$ (`expval` with `N_OP`).
4. Compute $\Delta x\,\Delta p$ (with `X_OP`, `P_OP`) and confirm it equals $\hbar/2$ — a coherent
   state is, like the ground state, minimum-uncertainty (in fact it is a *displaced* ground
   state). Frame: the most classical quantum state.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    [np.max(np.abs(a @ psi_alpha - alpha * psi_alpha)), dx_a * dp_a],
    [0.0, HBAR / 2],
    "a coherent state is a minimum-uncertainty eigenstate of the lowering operator (a|α⟩=α|α⟩, Δx·Δp=ℏ/2)",
    atol=1e-3,
)

## Exercise 8 — Coherent-state dynamics: a quantum particle that moves classically *(student)*

Evolve a coherent state and show its center oscillates *exactly* like a classical particle,
$\langle x\rangle(t)=\sqrt{2\hbar/m\omega}\,\alpha\cos\omega t$, while its width stays minimal
($\Delta x\,\Delta p=\hbar/2$ for all $t$ — the packet does not spread) {eq}`eq-coherent`.

1. Evolve $|\alpha\rangle$ under $H$ in the number basis: each component picks up a phase,
   $|\psi(t)\rangle=\sum_n c_n e^{-iE_nt/\hbar}|n\rangle$ (equivalently
   `scipy.linalg.expm(-1j*H*t) @ psi`).
2. Compute $\langle x\rangle(t)$ with `expval` and `X_OP` over a period.
3. Confirm it equals the classical trajectory $\sqrt{2\hbar/m\omega}\,\alpha\cos\omega t$.
4. Confirm $\Delta x\, \Delta p=\hbar/2$ is *maintained* for all $t$ — unique to the oscillator
   (the free particle's packet spreads, [§6.13](scattering-tunneling.ipynb)). Frame: the bridge from quantum to classical motion,
   and the state of a laser.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    x_quantum,
    x_classical,
    "a coherent state oscillates like a classical particle, ⟨x⟩(t)=√(2ℏ/mω)α cos ωt, and stays minimum-uncertainty for all t",
    atol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — The universal oscillator

The harmonic oscillator is the system all of physics reduces to near a minimum, and we solved it three
ways that had to agree — by an **algebra** that produced the entire equally-spaced ladder from the
single commutator $[a,a^{\dagger}]=1$, by **Hermite functions**, and on the **grid**. Its **zero-point
energy** $\hbar\omega/2$ is the uncertainty principle made energetic; its high states recover the
**classical** motion through the correspondence principle; and its **coherent states** oscillate like a
classical particle while staying as sharp as quantum mechanics permits. The ladder operators built here
are not a trick for one problem — they return for **angular momentum** ([§6.14](angular-momentum-algebra.ipynb)), and in Volume VII they
will count **photons**.

**This exercise (synthesis).** No new computation: the convergence of the three routes *is* the result.
Of all the systems in this volume, this is the one met most often — in every molecule's vibrations,
every solid's heat capacity, every mode of light. It is worth knowing three ways, because each is the
natural language in a different setting: the algebra for fields and angular momentum, the Hermite
functions for analytic work, the grid for any potential that is only *approximately* harmonic. The next
notebook ([§6.13](scattering-tunneling.ipynb)) sets wave packets in motion through barriers, where — unlike the rigid coherent state —
they spread and tunnel.

## Notebook summary

The oscillator solved three ways, with its zero-point energy, classical limit, and coherent states.

- **The ladder algebra** {eq}`eq-ladder`: $a,a^{\dagger}$ via `numpy.diag` ($\sqrt n$ off-diagonal),
  $[a,a^{\dagger}]=1$ (away from truncation), $N=a^{\dagger}a$ — the spectrum $E_n=\hbar\omega(n+\tfrac
  12)$ from the algebra alone.
- **Three routes agree** {eq}`eq-three-routes`: ladder, Hermite functions
  (`numpy.polynomial.hermite.hermval`, `math.factorial`), and the [§6.10](schrodinger-on-a-computer.ipynb) grid (`numpy.linalg.eigh`) give
  the same spectrum and states.
- **Zero-point energy** {eq}`eq-zero-point`: $E_0=\hbar\omega/2$, the ground state minimum-uncertainty —
  the uncertainty principle made energetic.
- **Correspondence** {eq}`eq-classical-quantum`: the quantum oscillator (discrete, zero-point) differs
  from Volume V's classical one (continuous, equipartition), but the smoothed high-$n$ density
  approaches the classical $1/\sqrt{A^2-x^2}$.
- **Coherent states** {eq}`eq-coherent`: $a|\alpha\rangle=\alpha|\alpha\rangle$, minimum-uncertainty,
  oscillating rigidly as $\langle x\rangle(t)=\sqrt{2\hbar/m\omega}\,\alpha\cos\omega t$ without
  spreading — the bridge to classical motion.

Every potential near a minimum is this one, and it is worth knowing all three ways — algebra, Hermite
functions, and the grid.

## Outlook

- **The same ladder algebra for angular momentum ([§6.14](angular-momentum-algebra.ipynb))** and, in Volume VII, for **photons** and
  quantum fields.
- **Wave-packet dynamics, spreading, and tunnelling ([§6.13](scattering-tunneling.ipynb))** — the contrast to the non-spreading
  coherent state.
- **The oscillator as the photon**: quantized field modes, the Casimir effect, the laser (Volume VII;
  horizons). **Squeezed states** (a horizon, named).
- **Cross-reference** [§6.9](position-representation.ipynb) (the Gaussian, uncertainty, $[x,p]=i\hbar$), [§6.10](schrodinger-on-a-computer.ipynb) (the grid solver), [§6.6](pauli-uncertainty.ipynb)
  (uncertainty), Volume V (the classical oscillator, equipartition), and forward to [§6.13](scattering-tunneling.ipynb), [§6.14](angular-momentum-algebra.ipynb), Volume
  VII.

In [ ]:
from ecp.style import footer

footer()